In [1]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler,StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score, cross_val_predict, GridSearchCV
from sklearn.pipeline import Pipeline



In [22]:
def cargar_y_preparar_datos(input):
    # Leer el archivo, eliminar filas vacías y transponer
    DE_df = pd.read_csv(input, sep=",", index_col=0)
    
    # Filtrar genes con nombres faltantes (índice vacío o NaN)
    DE_df = DE_df[~DE_df.index.isnull() & (DE_df.index != '')]
    
    # Transponer: filas = muestras, columnas = genes
    DE_df = DE_df.transpose()

    # Definir etiquetas
    labels = {"WT": 0, "TLR4": 1}

    # Extraer la etiqueta de cada muestra usando el penúltimo fragmento del nombre
    Y = [labels[i.split("_")[-2]] for i in DE_df.index]

    # Agregar columna 'Y' con las etiquetas
    DE_df["Y"] = Y

    return DE_df


def test_model_params(dataset, params, model, logfc_table, n_features = None):
    data_shuffled = dataset.sample(frac = 1)
    labels = data_shuffled["Y"]
    steps = list()
    steps.append(('scaler', MinMaxScaler()))
    data_shuffled = data_shuffled.drop("Y", axis = 1)[logfc_table[:n_features]]
    steps.append(('model', model))
    pipeline = Pipeline(steps=steps)
    search = GridSearchCV(pipeline, param_grid = params, cv = 2, n_jobs = 12)
    search.fit(data_shuffled, labels)
    return search


def test_feature_num(dataset, params, model, feat_range, logfc_table):
    df = pd.DataFrame()
    for feat in feat_range:
        res = test_model_params(dataset, params, model, logfc_table, feat)
        new_df = pd.DataFrame(res.cv_results_["params"])
        new_df["test_error"] = (1-res.cv_results_["mean_test_score"])*100
        new_df["features"] = [feat]*(new_df.shape[0])
        df = pd.concat([df, new_df])
    return df


In [19]:
WT_TLR4 = cargar_y_preparar_datos("WT_TLR4.csv")
print(WT_TLR4.head())

Gene                          LOC145474     LINP1  LINC01512  LINC02381  \
WT_1.gProcessedSignal          4.588821  7.166882   6.080101   5.856952   
WT_2.gProcessedSignal          5.960011  7.771754   5.870934   5.452845   
RNA5_TLR4_1.gProcessedSignal   6.599604  5.996803   5.576464   6.427738   
RNA5_TLR4_2.gProcessedSignal   6.413724  5.508821   5.237739   6.343293   

Gene                          LINC01122  LOC100130691  XLOC_l2_015760  \
WT_1.gProcessedSignal         12.512100      5.747722        8.218339   
WT_2.gProcessedSignal         12.387483      5.909303        8.612909   
RNA5_TLR4_1.gProcessedSignal  12.546988      6.648390        9.973552   
RNA5_TLR4_2.gProcessedSignal  12.761979      6.772706        9.852548   

Gene                          MIR100HG     HIPK2  LOC101927354  ...  \
WT_1.gProcessedSignal         4.966560  8.233332      7.103016  ...   
WT_2.gProcessedSignal         5.014599  8.108510      6.495858  ...   
RNA5_TLR4_1.gProcessedSignal  5.234023  7.786

In [20]:
DE_WT_TLR4 = pd.read_csv("DGE_TLR4_vs_WT.csv", sep=";", index_col=0)
rnk = list(DE_WT_TLR4.index)

In [ ]:
rango_C = [3.3]
rango_g = [k/100 for k in range(22,34,1)]
params = {"model__C": rango_C, "model__kernel": ["rbf"], "model__gamma": rango_g}

average_results =pd.DataFrame()
for i in range(1000):
    print(i)
    res = test_feature_num(WT_TLR4, params, SVC(), range(145, 161, 1), rnk)
    if i==0:
        average_results = res
    else:
        average_results[f'test_error_{i}'] = res["test_error"]



In [25]:
average_results['average_error'] = average_results.filter(like='test_error').mean(axis=1)
average_results['std_error'] = average_results.filter(like='test_error').std(axis=1)

average_results.to_csv("resultados_svm.csv")